# BMIF 802 Lab 7: Classification 2

## Tuning, Neural Networks, and Model Explanation

In this lab you will continue working with the Cleveland Heart Disease dataset from Lab 6. You will reuse your preprocessing work, add an SVM classifier, tune models, build a small fully connected neural network, and use SHAP values to help explain the neural network output.

## 1. Setup

Import the packages you need. You may need to install `tensorflow` and `shap` if they are not already in your environment.

For Google Colab, you can run this in a code cell if needed:

```python
!pip install shap
```

TensorFlow/Keras is usually already available in Colab.


In [2]:
# Core packages
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# scikit-learn tools
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    roc_auc_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    RocCurveDisplay,
)

# Models from Lab 6 and the new SVM model
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC

# Parameter distributions for random search
from scipy.stats import randint, loguniform

# Neural network packages
%pip install tensorflow
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# This keeps small CPU-based neural network examples running smoothly on laptops.
try:
    tf.config.threading.set_intra_op_parallelism_threads(1)
    tf.config.threading.set_inter_op_parallelism_threads(1)
except RuntimeError:
    pass

# SHAP is used near the end of the lab
# import shap


   ---------------------------------------- 0.0/350.9 MB ? eta -:--:--
   ---------------------------------------- 0.8/350.9 MB 4.2 MB/s eta 0:01:24
   ---------------------------------------- 1.3/350.9 MB 3.9 MB/s eta 0:01:29
   ---------------------------------------- 2.9/350.9 MB 4.7 MB/s eta 0:01:15
   ---------------------------------------- 4.2/350.9 MB 5.2 MB/s eta 0:01:07
    --------------------------------------- 5.8/350.9 MB 5.5 MB/s eta 0:01:03
    --------------------------------------- 7.1/350.9 MB 5.7 MB/s eta 0:01:01
    --------------------------------------- 8.1/350.9 MB 5.7 MB/s eta 0:01:01
   - -------------------------------------- 9.4/350.9 MB 5.6 MB/s eta 0:01:01
   - -------------------------------------- 10.7/350.9 MB 5.7 MB/s eta 0:01:00
   - -------------------------------------- 12.1/350.9 MB 5.8 MB/s eta 0:00:59
   - -------------------------------------- 13.6/350.9 MB 5.9 MB/s eta 0:00:58
   - -------------------------------------- 15.5/350.9 MB 6.2 MB/s e

## 2. Load the dataset

Use the same Cleveland Heart Disease dataset from Lab 6.

Expected location:

```text
data/heart_disease_cleveland.csv
```

Tasks:

1. Load the data.
2. Replace any `?` values with missing values.
3. Confirm the rows and columns look correct.
4. Create a binary target column called `heart_disease` from the original `num` column.

In [3]:
# TODO: Load the Cleveland Heart Disease dataset.
data_path = "data/heart_disease_cleveland.csv"
data = pd.read_csv(data_path, na_values="?")

# Suggested column names if you need them:
columns = [
    "age", "sex", "cp", "trestbps", "chol", "fbs", "restecg", "thalach",
    "exang", "oldpeak", "slope", "ca", "thal", "num"
]

# TODO: Read the CSV file into a DataFrame.
# data = ...

# TODO: Replace ? with np.nan if needed.

# TODO: Create a binary target column.
# data["heart_disease"] = ...
data["heart_disease"] = (data["num"] > 0).astype(int)

# TODO: Display the first few rows.
data.head()

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,num,heart_disease
0,63.0,1.0,1.0,145.0,233.0,1.0,2.0,150.0,0.0,2.3,3.0,0.0,6.0,0,0
1,67.0,1.0,4.0,160.0,286.0,0.0,2.0,108.0,1.0,1.5,2.0,3.0,3.0,2,1
2,67.0,1.0,4.0,120.0,229.0,0.0,2.0,129.0,1.0,2.6,2.0,2.0,7.0,1,1
3,37.0,1.0,3.0,130.0,250.0,0.0,0.0,187.0,0.0,3.5,3.0,0.0,3.0,0,0
4,41.0,0.0,2.0,130.0,204.0,0.0,2.0,172.0,0.0,1.4,1.0,0.0,3.0,0,0


## 3. Reuse your Lab 6 preprocessing

Before training models, rebuild the preprocessing workflow from Lab 6.

For this lab, your preprocessing should handle:

- Numeric features.
- Categorical features.
- Missing values.
- Feature scaling.
- One-hot encoding.

Important: SVMs and neural networks are sensitive to feature scaling, so numeric variables should be standardized.

In [5]:
# TODO: Define X and y.
X = data.drop(columns=["num", "heart_disease"])
y = data["heart_disease"]

# TODO: Split into training and test sets.
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# TODO: Optional, create a validation set from the training data for the neural network.
# X_train, X_val, y_train, y_val = train_test_split(...)

# TODO: List your numeric and categorical features.
# numeric_features = [...]
# categorical_features = [...]

# TODO: Build your preprocessing pipeline.
# numeric_transformer = ...
# categorical_transformer = ...
# preprocessor = ...

X_train = X_train_raw.copy()
X_test = X_test_raw.copy()

imputation_values = {}
for column in ["ca", "thal"]:
    training_mode = X_train_raw[column].mode().iloc[0]
    imputation_values[column] = training_mode
    X_train[column] = X_train[column].fillna(training_mode)
    X_test[column] = X_test[column].fillna(training_mode)

print("Training-set imputation values:")
print(imputation_values)

categorical_columns = ["cp", "restecg", "slope", "thal"]
X_train = pd.get_dummies(X_train, columns=categorical_columns, dtype=int)
X_test = pd.get_dummies(X_test, columns=categorical_columns, dtype=int)

# Ensure the testing matrix has the same columns in the same order.
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)


Training-set imputation values:
{'ca': np.float64(0.0), 'thal': np.float64(3.0)}


## 4. Evaluation helper

Create a helper function that calculates several classification metrics.

Include at least:

- Accuracy
- Precision
- Recall
- F1-score
- MCC
- ROC AUC

In [6]:
# TODO: Write a function that evaluates one fitted classifier.

def evaluate_model(name, model, X_test, y_test):
    
    y_pred = model.predict(X_test)

    metrics = {
        "model": name,
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred, zero_division=0),
        "f1_score": f1_score(y_test, y_pred, zero_division=0),
        "mcc": matthews_corrcoef(y_test, y_pred),
    }

    # ROC AUC (binary classification)
    try:
        if hasattr(model, "predict_proba"):
            y_score = model.predict_proba(X_test)[:, 1]
            metrics["roc_auc"] = roc_auc_score(y_test, y_score)
        elif hasattr(model, "decision_function"):
            y_score = model.decision_function(X_test)
            metrics["roc_auc"] = roc_auc_score(y_test, y_score)
        else:
            metrics["roc_auc"] = None
    except Exception:
        metrics["roc_auc"] = None

    return metrics

## 5. Reuse the models from Lab 6

Train the same classification models that you used in Lab 6.

In [ ]:
# TODO: Rebuild and train your Lab 6 models using the preprocessing pipeline.

# baseline_models = {
#     "Logistic Regression": ...,
#     "KNN": ...,
#     "Decision Tree": ...,
#     "Random Forest": ...,
# }
results=[]

# baseline
from sklearn.dummy import DummyClassifier
baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)

results.append(
    evaluate_model(
        "Majority Class Baseline",
        baseline,
        X_test,
        y_test,
    )
)

# KNN
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

knn_scaled_5 = KNeighborsClassifier(n_neighbors=17)
knn_scaled_5.fit(X_train_scaled, y_train)

knn_scaled_5_train_accuracy = knn_scaled_5.score(X_train_scaled, y_train)
knn_scaled_5_test_accuracy = knn_scaled_5.score(X_test_scaled, y_test)

knn_final = KNeighborsClassifier(n_neighbors=5)
knn_final.fit(X_train_scaled, y_train)

results.append( evaluate_model(
    "k-NN",
    knn_final,
    X_test_scaled,
    y_test
))



# Descision Tree
decision_tree = DecisionTreeClassifier(random_state=42)
decision_tree.fit(X_train, y_train)

results.append(
    evaluate_model(
        "Decision Tree",
        decision_tree,
        X_test,
        y_test
    )
)

# Random Forest
random_forest = RandomForestClassifier(random_state=42)
random_forest.fit(X_train, y_train)

results.append(
    evaluate_model(
        "RandomForest",
        random_forest,
        X_test,
        y_test
    )
)

# TODO: Store the results in a DataFrame.
results = pd.DataFrame(results)

<bound method NDFrame.head of                      model  accuracy  precision    recall  f1_score       mcc  \
0  Majority Class Baseline  0.540984   0.000000  0.000000  0.000000  0.000000   
1                     k-NN  0.852459   0.787879  0.928571  0.852459  0.716450   
2            Decision Tree  0.754098   0.685714  0.857143  0.761905  0.527823   
3             RandomForest  0.868852   0.833333  0.892857  0.862069  0.738947   

    roc_auc  
0  0.500000  
1  0.929113  
2  0.761905  
3  0.929113  >


In [10]:
print(results)

                     model  accuracy  precision    recall  f1_score       mcc  \
0  Majority Class Baseline  0.540984   0.000000  0.000000  0.000000  0.000000   
1                     k-NN  0.852459   0.787879  0.928571  0.852459  0.716450   
2            Decision Tree  0.754098   0.685714  0.857143  0.761905  0.527823   
3             RandomForest  0.868852   0.833333  0.892857  0.862069  0.738947   

    roc_auc  
0  0.500000  
1  0.929113  
2  0.761905  
3  0.929113  


## 6. Add Support Vector Machine

Train an SVM classifier.

Try at least one of these:

- Linear kernel
- RBF kernel

Questions to think about:

1. Why does SVM need scaled numeric features?
2. What does the kernel change?
3. How does the SVM compare with your Lab 6 models?

In [ ]:
# TODO: Add an SVM model.

# svm_model = Pipeline([
#     ("preprocess", preprocessor),
#     ("model", SVC(kernel="rbf", probability=True, random_state=42))
# ])

# TODO: Fit and evaluate the SVM.

## 7. Confusion matrix and ROC curve

Choose at least two models and display:

- A confusion matrix.
- A ROC curve.

Write a short interpretation. For example, does one model have better recall? Does one model have better precision? Is the AUC different?

In [ ]:
# TODO: Plot confusion matrices and ROC curves for selected models.

## 8. Random Forest parameter tuning

Use both grid search and random search for Random Forest.

Tune more than one parameter. Examples:

- `n_estimators`
- `max_depth`
- `min_samples_split`
- `min_samples_leaf`
- `max_features`

Compare the best grid search model and best random search model on the test set.

In [ ]:
# TODO: Grid search for Random Forest.

# rf_grid = {
#     "model__n_estimators": [...],
#     "model__max_depth": [...],
#     "model__min_samples_leaf": [...],
# }

# TODO: Random search for Random Forest.

# rf_random = {
#     "model__n_estimators": randint(...),
#     "model__max_depth": [...],
#     "model__min_samples_split": randint(...),
#     "model__min_samples_leaf": randint(...),
# }

## 9. SVM parameter tuning

Use both grid search and random search for SVM.

Tune more than one parameter. Examples:

- `C`
- `kernel`
- `gamma`

Compare the best grid search model and best random search model on the test set.

In [ ]:
# TODO: Grid search for SVM.

# svm_grid = {
#     "model__kernel": [...],
#     "model__C": [...],
#     "model__gamma": [...],
# }

# TODO: Random search for SVM.

# svm_random = {
#     "model__C": loguniform(...),
#     "model__gamma": loguniform(...),
#     "model__kernel": [...],
# }

## 10. Fully connected neural network with Keras

You will now build a small fully connected neural network.

The network should have:

- An input layer matching the number of preprocessed features.
- At least one hidden layer.
- A ReLU activation.
- An output layer with one output value.

For binary classification in Keras, it is common to use a sigmoid output layer and train with binary cross-entropy loss.


In [ ]:
# TODO: Fit your preprocessor on the training data and transform train/validation/test data.

# X_train_nn = ...
# X_val_nn = ...
# X_test_nn = ...

# TODO: Convert the target values into NumPy arrays.

# TODO: Define a small fully connected neural network using keras.Sequential.

# model = keras.Sequential([
#     keras.Input(shape=(X_train_nn.shape[1],)),
#     layers.Dense(..., activation="relu"),
#     layers.Dropout(...),
#     layers.Dense(1, activation="sigmoid"),
# ])

# TODO: Compile the model using binary_crossentropy loss and the Adam optimizer.


## 11. Train the neural network

Train the model on your laptop CPU.

Keep the model small so that it runs quickly:

- 1 hidden layer is enough.
- 20 to 50 epochs is enough.
- Use a small batch size such as 16 or 32.

Track both training loss and validation loss.

In [ ]:
# TODO: Write code to train the Keras model.

# Tips:
# - Use binary_crossentropy loss.
# - Use Adam optimizer.
# - Use model.fit(...).
# - Use a validation set to choose the best model.
# - Use an EarlyStopping callback to stop training when validation loss stops improving.


## 12. Tune the neural network

Try a small parameter search. Keep it small enough to run on a laptop.

Tune at least three settings, such as:

- Number of hidden units.
- Learning rate.
- Dropout rate.
- Batch size.
- Weight decay.

Create a small table of validation results and choose the best setting.

In [ ]:
# TODO: Try several neural network parameter settings.

# configs = [
#     {"hidden_units": 16, "learning_rate": 0.001, "dropout": 0.0, "batch_size": 32},
#     ...
# ]

# TODO: Train one model per config and compare validation AUC or validation F1-score.

## 13. Run the neural network in Google Colab

Upload this notebook and the dataset to Google Colab.

Then use:

**Runtime → Change runtime type → Hardware accelerator → GPU**

Run the following cell to check whether TensorFlow/Keras sees a GPU.


In [ ]:
gpus = tf.config.list_physical_devices("GPU")
print("GPUs detected:", gpus)

if len(gpus) > 0:
    print("TensorFlow/Keras can use a GPU in this runtime.")
else:
    print("No GPU detected. The model will run on CPU, which is fine for this small dataset.")


## 14. SHAP values for neural network explanation

Use SHAP values to explain the neural network output.

To keep runtime reasonable:

- Use a small background sample from the training set.
- Explain only a small sample of test rows.
- Start with around 20 to 50 background rows and 10 to 25 explanation rows.

Questions:

1. Which features appear most important?
2. Do important features make clinical sense?
3. Are the important features the same as for your Random Forest model?
4. How does one-hot encoding affect the feature names?

In [ ]:
# TODO: Import shap and explain the neural network predictions.

# import shap

# TODO: Create a small background sample.
# TODO: Create a small test sample to explain.
# TODO: Build a prediction function that returns neural network probabilities.
# TODO: Create a SHAP explainer.
# TODO: Plot a SHAP summary plot.